In [2]:
import pandas as pd
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
#pytorch dataset
class FakeNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [4]:
def load_and_tokenisation(dataset_path):
  # ============================================================================
  # 2. Load and Prepare Data
  # ============================================================================
  print("\n" + "=" * 60)
  print("DATA LOADING")
  print("=" * 60)

  df = pd.read_csv(dataset_path).dropna()
  print(f"✓ Dataset loaded: {len(df)} rows")
  print(f"✓ Columns: {df.columns.tolist()}")
  print(f"\nLabel distribution:")
  print(df['label'].value_counts())

  # ============================================================================
  # 3. Train/Validation/Test Split
  # ============================================================================
  print("\n" + "=" * 60)
  print("DATA SPLITTING")
  print("=" * 60)

  # First split: 70% train, 30% temp (for val + test)
  train_texts, temp_texts, train_labels, temp_labels = train_test_split(
      df['combined_text'].tolist(),
      df['label'].tolist(),
      test_size=0.3, # Changed to 0.3 to get 70% train
      random_state=42,
      stratify=df['label']
  )

  # Second split: 50% validation, 50% test from the 30% temp data (which means 15% val, 15% test from total)
  val_texts, test_texts, val_labels, test_labels = train_test_split(
      temp_texts,
      temp_labels,
      test_size=0.5,
      random_state=42,
      stratify=temp_labels
  )

  print(f"✓ Training samples: {len(train_texts)}")
  print(f"✓ Validation samples: {len(val_texts)}")
  print(f"✓ Test samples: {len(test_texts)}")

  # ============================================================================
  # 4. Tokenization
  # ============================================================================
  print("\n" + "=" * 60)
  print("TOKENIZATION")
  print("=" * 60)

  tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
  print("✓ Tokenizer loaded")

  print("Tokenizing training data...")
  train_encodings = tokenizer(
      train_texts,
      truncation=True,
      padding=True,
      max_length=128
  )
  print("✓ Training data tokenized")

  print("Tokenizing validation data...")
  val_encodings = tokenizer(
      val_texts,
      truncation=True,
      padding=True,
      max_length=128
  )
  print("✓ Validation data tokenized")

  print("Tokenizing test data...")
  test_encodings = tokenizer(
      test_texts,
      truncation=True,
      padding=True,
      max_length=128
  )
  print("✓ Test data tokenized")

  # ============================================================================
  # 5. Create PyTorch Dataset
  # ============================================================================
  print("\n" + "=" * 60)
  print("DATASET CREATION")
  print("=" * 60)

  train_dataset = FakeNewsDataset(train_encodings, train_labels)
  val_dataset = FakeNewsDataset(val_encodings, val_labels)
  test_dataset = FakeNewsDataset(test_encodings, test_labels)

  print(f"✓ Train dataset: {len(train_dataset)} samples")
  print(f"✓ Validation dataset: {len(val_dataset)} samples")
  print(f"✓ Test dataset: {len(test_dataset)} samples")

  return train_dataset, val_dataset, test_dataset, tokenizer

In [5]:
def load_model():
  # ============================================================================
  # 1. Setup Device (TPU/CUDA/CPU)
  # ============================================================================
  print("=" * 60)
  print("DEVICE SETUP")
  print("=" * 60)

  # Check for TPU first
  try:
      import torch_xla.core.xla_model as xm
      device = xm.xla_device()
      print(f"✓ Using TPU: {device}")
      use_tpu = True
  except ImportError:
      use_tpu = False
      if torch.cuda.is_available():
          device = torch.device("cuda")
          print(f"✓ Using CUDA: {torch.cuda.get_device_name(0)}")
          print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
      else:
          device = torch.device("cpu")
          print("⚠ Using CPU (Training will be slow!)")

  # ============================================================================
  # 2. Load Pre-trained BERT Model
  # ============================================================================
  print("\n" + "=" * 60)
  print("MODEL LOADING")
  print("=" * 60)

  model = BertForSequenceClassification.from_pretrained(
      'bert-base-uncased',
      num_labels=2
  )

  if not use_tpu:
      model.to(device)
      print(f"✓ BERT model loaded and moved to {device}")
  else:
      print("✓ BERT model loaded (TPU will handle device placement)")

  return model, device, use_tpu

In [6]:
# ============================================================================
# 7. Define Metrics
# ============================================================================
def compute_metrics(pred):
    """Compute accuracy, precision, recall, F1"""
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


In [7]:
def finetuning(model, train_dataset, val_dataset, test_dataset, compute_metrics_fn, use_tpu, device):
  print("\n" + "=" * 60)
  print("TRAINING CONFIGURATION")
  print("=" * 60)

  # 8. Define Training Arguments
  training_args = TrainingArguments(
      output_dir='./results',
      num_train_epochs=3,
      per_device_train_batch_size=32,
      per_device_eval_batch_size=64,
      warmup_steps=500,
      weight_decay=0.01,
      logging_dir='./logs',
      logging_steps=50,
      eval_strategy="steps",
      eval_steps=1000,
      save_strategy="steps",
      save_steps=1000,
      save_total_limit=2,
      load_best_model_at_end=True,
      metric_for_best_model="f1",
      greater_is_better=True,
      optim="adamw_torch",
      fp16=False,
      dataloader_num_workers=0,
      report_to="none",
  )

  print("Training configuration:")
  print(f"  - Epochs: {training_args.num_train_epochs}")
  print(f"  - Train batch size: {training_args.per_device_train_batch_size}")
  print(f"  - Eval batch size: {training_args.per_device_eval_batch_size}")
  print(f"  - FP16: {training_args.fp16}")
  print(f"  - Device: {'TPU' if use_tpu else device}")


  print("\n" + "=" * 60)
  print("TRAINER INITIALIZATION")
  print("=" * 60)

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=val_dataset, # The eval_dataset here is for validation during training
      compute_metrics=compute_metrics_fn
  )
  print("✓ Trainer initialized")

  print("\n" + "=" * 60)
  print("TRAINING START")
  print("=" * 60)

  trainer.train()
  print("\n✓ Training completed!")

  print("\n" + "=" * 60)
  print("VALIDATION SET EVALUATION")
  print("=" * 60)

  val_results = trainer.evaluate()
  print("Validation Results:")
  for key, value in val_results.items():
      print(f"  {key}: {value:.4f}")

  print("\n" + "=" * 60)
  print("TEST SET EVALUATION")
  print("=" * 60)

  test_results = trainer.evaluate(eval_dataset=test_dataset)
  print("Test Results:")
  for key, value in test_results.items():
      print(f"  {key}: {value:.4f}")

In [8]:
def save_model(model, tokenizer, output_path, use_tpu, device):
  print("\n" + "=" * 60)
  print("MODEL SAVING")
  print("=" * 60)

  output_dir = output_path

  # Move model to CPU before saving if it's on TPU
  if use_tpu:
      model.to("cpu")
      print("Moved model to CPU for saving.")

  model.save_pretrained(output_dir)
  tokenizer.save_pretrained(output_dir)
  print(f"✓ Model saved to {output_dir}")

  # Move model back to original device after saving
  if use_tpu:
      model.to(device)
      print("Moved model back to TPU.")

  print("\n" + "=" * 60)
  print("BERT FINE-TUNING COMPLETE! 🎉")
  print("=" * 60)

In [9]:
def train_loop(dataset_path, output_path):
  train_dataset, val_dataset, test_dataset, tokenizer = load_and_tokenisation(dataset_path)
  model, device, use_tpu = load_model()
  finetuning(model, train_dataset, val_dataset, test_dataset, compute_metrics, use_tpu, device)
  save_model(model, tokenizer, output_path, use_tpu, device)

# WELFake Dataset

In [11]:
train_loop("/content/drive/MyDrive/datasets/WELFake_processed.csv", "/content/drive/MyDrive/bert_models/WELFake")


DATA LOADING
✓ Dataset loaded: 63670 rows
✓ Columns: ['combined_text', 'label']

Label distribution:
label
0    34790
1    28880
Name: count, dtype: int64

DATA SPLITTING
✓ Training samples: 44569
✓ Validation samples: 9550
✓ Test samples: 9551

TOKENIZATION


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...


✓ Training completed!

VALIDATION SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation Results:
  eval_loss: 0.0423
  eval_accuracy: 0.9923
  eval_f1: 0.9915
  eval_precision: 0.9879
  eval_recall: 0.9952
  eval_runtime: 4.9289
  eval_samples_per_second: 973.8390
  eval_steps_per_second: 15.2160
  epoch: 3.0000

TEST SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test Results:
  eval_loss: 0.0543
  eval_accuracy: 0.9898
  eval_f1: 0.9889
  eval_precision: 0.9844
  eval_recall: 0.9933
  eval_runtime: 12.7785
  eval_samples_per_second: 375.6310
  eval_steps_per_second: 5.8690
  epoch: 3.0000

MODEL SAVING
Moved model to CPU for saving.
✓ Model saved to /content/drive/MyDrive/bert_models/WELFake
Moved model back to TPU.

BERT FINE-TUNING COMPLETE! 🎉


# FakeNewsNet

In [12]:
train_loop("/content/drive/MyDrive/datasets/FakeNewsNet_processed.csv", "/content/drive/MyDrive/bert_models/FakeNewsNet")


DATA LOADING
✓ Dataset loaded: 21844 rows
✓ Columns: ['combined_text', 'label']

Label distribution:
label
1    16522
0     5322
Name: count, dtype: int64

DATA SPLITTING
✓ Training samples: 15290
✓ Validation samples: 3277
✓ Test samples: 3277

TOKENIZATION
✓ Tokenizer loaded
Tokenizing training data...
✓ Training data tokenized
Tokenizing validation data...
✓ Validation data tokenized
Tokenizing test data...


/tmp/ipython-input-1204052393.py:12: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Test data tokenized

DATASET CREATION
✓ Train dataset: 15290 samples
✓ Validation dataset: 3277 samples
✓ Test dataset: 3277 samples
DEVICE SETUP
✓ Using TPU: xla:0

MODEL LOADING
✓ BERT model loaded (TPU will handle device placement)

TRAINING CONFIGURATION
Training configuration:
  - Epochs: 3
  - Train batch size: 32
  - Eval batch size: 64
  - FP16: False
  - Device: TPU

TRAINER INITIALIZATION
✓ Trainer initialized

TRAINING START


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1000,0.193500,0.472684,0.841013,0.896029,0.886651,0.905607



✓ Training completed!

VALIDATION SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation Results:
  eval_loss: 0.4727
  eval_accuracy: 0.8410
  eval_f1: 0.8960
  eval_precision: 0.8867
  eval_recall: 0.9056
  eval_runtime: 1.3920
  eval_samples_per_second: 1195.3660
  eval_steps_per_second: 18.6780
  epoch: 3.0000

TEST SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test Results:
  eval_loss: 0.4635
  eval_accuracy: 0.8499
  eval_f1: 0.9016
  eval_precision: 0.8934
  eval_recall: 0.9100
  eval_runtime: 18.5914
  eval_samples_per_second: 89.5040
  eval_steps_per_second: 1.3980
  epoch: 3.0000

MODEL SAVING
Moved model to CPU for saving.
✓ Model saved to /content/drive/MyDrive/bert_models/FakeNewsNet
Moved model back to TPU.

BERT FINE-TUNING COMPLETE! 🎉


# Fake News Detection Dataset

In [13]:
train_loop("/content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv", "/content/drive/MyDrive/bert_models/Fake_News_Detection")


DATA LOADING
✓ Dataset loaded: 38650 rows
✓ Columns: ['combined_text', 'label']

Label distribution:
label
1    21194
0    17456
Name: count, dtype: int64

DATA SPLITTING
✓ Training samples: 27055
✓ Validation samples: 5797
✓ Test samples: 5798

TOKENIZATION
✓ Tokenizer loaded
Tokenizing training data...
✓ Training data tokenized
Tokenizing validation data...
✓ Validation data tokenized
Tokenizing test data...
✓ Test data tokenized

DATASET CREATION
✓ Train dataset: 27055 samples
✓ Validation dataset: 5797 samples
✓ Test dataset: 5798 samples
DEVICE SETUP
✓ Using TPU: xla:0

MODEL LOADING


/tmp/ipython-input-1204052393.py:12: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ BERT model loaded (TPU will handle device placement)

TRAINING CONFIGURATION
Training configuration:
  - Epochs: 3
  - Train batch size: 32
  - Eval batch size: 64
  - FP16: False
  - Device: TPU

TRAINER INITIALIZATION
✓ Trainer initialized

TRAINING START


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1000,0.016600,0.013096,0.996032,0.996379,0.997478,0.995282
2000,0.000100,0.009315,0.997585,0.997801,0.996548,0.999056


...


✓ Training completed!

VALIDATION SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation Results:
  eval_loss: 0.0093
  eval_accuracy: 0.9976
  eval_f1: 0.9978
  eval_precision: 0.9965
  eval_recall: 0.9991
  eval_runtime: 3.0752
  eval_samples_per_second: 946.9270
  eval_steps_per_second: 14.9580
  epoch: 3.0000

TEST SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test Results:
  eval_loss: 0.0128
  eval_accuracy: 0.9969
  eval_f1: 0.9972
  eval_precision: 0.9950
  eval_recall: 0.9994
  eval_runtime: 10.2519
  eval_samples_per_second: 284.0440
  eval_steps_per_second: 4.4870
  epoch: 3.0000

MODEL SAVING
Moved model to CPU for saving.
✓ Model saved to /content/drive/MyDrive/bert_models/Fake_News_Detection
Moved model back to TPU.

BERT FINE-TUNING COMPLETE! 🎉


# ISOT Dataset

In [14]:
train_loop("/content/drive/MyDrive/datasets/ISOT_processed.csv", "/content/drive/MyDrive/bert_models/ISOT")


DATA LOADING
✓ Dataset loaded: 39098 rows
✓ Columns: ['combined_text', 'label']

Label distribution:
label
0    21196
1    17902
Name: count, dtype: int64

DATA SPLITTING
✓ Training samples: 27368
✓ Validation samples: 5865
✓ Test samples: 5865

TOKENIZATION
✓ Tokenizer loaded
Tokenizing training data...
✓ Training data tokenized
Tokenizing validation data...
✓ Validation data tokenized
Tokenizing test data...
✓ Test data tokenized

DATASET CREATION
✓ Train dataset: 27368 samples
✓ Validation dataset: 5865 samples
✓ Test dataset: 5865 samples
DEVICE SETUP
✓ Using TPU: xla:0

MODEL LOADING


/tmp/ipython-input-1204052393.py:12: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ BERT model loaded (TPU will handle device placement)

TRAINING CONFIGURATION
Training configuration:
  - Epochs: 3
  - Train batch size: 32
  - Eval batch size: 64
  - FP16: False
  - Device: TPU

TRAINER INITIALIZATION
✓ Trainer initialized

TRAINING START


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


...


✓ Training completed!

VALIDATION SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation Results:
  eval_loss: 0.0025
  eval_accuracy: 0.9997
  eval_f1: 0.9996
  eval_precision: 0.9993
  eval_recall: 1.0000
  eval_runtime: 3.1308
  eval_samples_per_second: 940.3400
  eval_steps_per_second: 14.6930
  epoch: 3.0000

TEST SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test Results:
  eval_loss: 0.0031
  eval_accuracy: 0.9997
  eval_f1: 0.9996
  eval_precision: 0.9993
  eval_recall: 1.0000
  eval_runtime: 3.1144
  eval_samples_per_second: 945.2890
  eval_steps_per_second: 14.7700
  epoch: 3.0000

MODEL SAVING
Moved model to CPU for saving.
✓ Model saved to /content/drive/MyDrive/bert_models/ISOT
Moved model back to TPU.

BERT FINE-TUNING COMPLETE! 🎉


# Fake News Classification Dataset

In [15]:
train_loop("/content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv", "/content/drive/MyDrive/bert_models/Fake_News_Detection")


DATA LOADING
✓ Dataset loaded: 38650 rows
✓ Columns: ['combined_text', 'label']

Label distribution:
label
1    21194
0    17456
Name: count, dtype: int64

DATA SPLITTING
✓ Training samples: 27055
✓ Validation samples: 5797
✓ Test samples: 5798

TOKENIZATION
✓ Tokenizer loaded
Tokenizing training data...
✓ Training data tokenized
Tokenizing validation data...
✓ Validation data tokenized
Tokenizing test data...
✓ Test data tokenized

DATASET CREATION
✓ Train dataset: 27055 samples
✓ Validation dataset: 5797 samples
✓ Test dataset: 5798 samples
DEVICE SETUP
✓ Using TPU: xla:0

MODEL LOADING


/tmp/ipython-input-1204052393.py:12: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ BERT model loaded (TPU will handle device placement)

TRAINING CONFIGURATION
Training configuration:
  - Epochs: 3
  - Train batch size: 32
  - Eval batch size: 64
  - FP16: False
  - Device: TPU

TRAINER INITIALIZATION
✓ Trainer initialized

TRAINING START


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1000,0.023900,0.020329,0.994135,0.994664,0.992484,0.996854
2000,0.002500,0.009374,0.998275,0.998426,0.999370,0.997483


...


✓ Training completed!

VALIDATION SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation Results:
  eval_loss: 0.0094
  eval_accuracy: 0.9983
  eval_f1: 0.9984
  eval_precision: 0.9994
  eval_recall: 0.9975
  eval_runtime: 3.1090
  eval_samples_per_second: 936.6370
  eval_steps_per_second: 14.7960
  epoch: 3.0000

TEST SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test Results:
  eval_loss: 0.0150
  eval_accuracy: 0.9974
  eval_f1: 0.9976
  eval_precision: 0.9975
  eval_recall: 0.9978
  eval_runtime: 3.1813
  eval_samples_per_second: 915.3420
  eval_steps_per_second: 14.4590
  epoch: 3.0000

MODEL SAVING
Moved model to CPU for saving.
✓ Model saved to /content/drive/MyDrive/bert_models/Fake_News_Detection
Moved model back to TPU.

BERT FINE-TUNING COMPLETE! 🎉
